# Understanding Pytorch neural network components
![](https://drive.google.com/uc?id=1eE-Kdt8-MnkPHCCD74KBH0dMJUnubccg)

**Break down of a simple neural network**

$\boxed {Z = W_1 * X + b_1}$  
$\boxed {Z' = A(Z)}$  
$\boxed {Y = W_2 * Z + b_2}$  

Where:  
X $\to$ input  
$W_x$ $\to$ weights  
$b_x$ $\to$ bias  
$A$ $\to$ activation function  
$Y$ $\to$ output  


- Loss function
- Backpropagation
- Optimizer

---

## Components of pytorch

- **Base class for defining customer models**: `torch.nn.Module`
- **Fully connected (dense) layers**: `toch.nn.Linear`
- **Activation fucntion**: `torch.nn.ReLU`
- **Optimiser**: `torch.optim`
- **Loss function**: `torch.nn.CrossEntropyLoss`
- **Loads data in batch**: `torch.utils.data.DataLoader`

---

## Different ways to create neural network

1. Function : Flexible, harder to interpret

2. Sequential : `nn.Sequential` $\to$ stuctrual approach

---

# Building a neural network

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

**Create simple neural network**

In [2]:
# Define a neural network model
class SimpleNN(nn.Module): # SimpleNN is a child class of nn.Module.

    def __init__(self, input_size, hidden_size, output_size):
        # Before defining model's layers, initialize the parent class
        super(SimpleNN, self).__init__()

        # Fully connected layer: input features → hidden neurons
        self.fcl1 = nn.Linear(input_size, hidden_size)
        # Apply ReLU activation: replace negative values with zero
        self.relu = nn.ReLU()
        # Fully connected layer: hidden neurons → output values
        self.fcl2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):  # define the forward pass through the network
        x = self.fcl1(x)   # linear transformation: x @ weights.T + bias
        x = self.relu(x)   # add non-linearity
        x = self.fcl2(x)   # transform hidden features into output values
        return x           # return the model's raw output
    
# NOTE: In this mode, we can change the flow of each layer.

**Create simple neural network sequential**

In [33]:
# Define a neural network model  
class SimpleNNSequential(nn.Module): # SimpleNN is a child class of nn.Module.

    def __init__(self, input_size, hidden_size, output_size):
        # Before defining model's layers, initialize the parent class
        super(SimpleNN, self).__init__()

        # Define layers in order: input → hidden → ReLU → output
        self.network = nn.Sequential(nn.Linear(input_size, output_size),  # transform input features to hidden features
                                     nn.ReLU(),                           # apply non-linear activation
                                     nn.Linear(hidden_size, output_size)) # transform hidden features to output values
        

    def forward(self, x):   # define the forward pass through the network
        x = self.network(x) # pass input through all sequential layers
        return x            # return the model's raw output

# NOTE: In sequential mode, we cannot change the flow of each layer.

## Training the neural network

In [3]:
# Create functional model
func_model = SimpleNN(input_size=4, hidden_size=8, output_size=3)
print(func_model)

SimpleNN(
  (fcl1): Linear(in_features=4, out_features=8, bias=True)
  (relu): ReLU()
  (fcl2): Linear(in_features=8, out_features=3, bias=True)
)


In [4]:
next(func_model.parameters())

Parameter containing:
tensor([[ 0.0246,  0.0085, -0.3825, -0.1138],
        [ 0.4177, -0.1527, -0.2976, -0.0353],
        [-0.4252, -0.2055,  0.0449,  0.1531],
        [-0.0968, -0.0748, -0.4706, -0.4513],
        [-0.4474, -0.3856, -0.3286, -0.4402],
        [-0.4172, -0.0401,  0.2459,  0.2835],
        [ 0.0574, -0.1104, -0.2091,  0.4824],
        [ 0.2885,  0.2898,  0.2800, -0.0499]], requires_grad=True)

In [5]:
# Create dummy data
X = torch.randn(10, 4) # create 4 features because of input_size = 4
Y = torch.randint(low=0, high=3, size=(10,)) # create 10 outputs for 10 different values of inputs

criterion = nn.CrossEntropyLoss() # loss function
optimizer = optim.Adam(params=func_model.parameters(), lr=0.01)

In [6]:
print(X)

tensor([[-1.3445, -0.0417,  1.2063,  1.0015],
        [-0.3705,  0.0612, -0.0473,  0.4044],
        [ 0.4641,  0.5728,  0.2613, -1.0117],
        [-1.6554,  0.0764, -0.0476,  0.6285],
        [-0.9182,  1.8736,  1.3139,  0.1411],
        [-1.8018,  1.0848,  0.6908, -0.4399],
        [-2.0797,  0.7176, -1.2683, -0.9840],
        [ 1.5434,  0.1288, -0.1317, -0.1102],
        [ 0.0907, -0.6813,  0.6471,  0.1382],
        [ 1.2523,  0.7578, -1.1317, -0.9905]])


In [7]:
print(Y)

tensor([0, 0, 0, 2, 0, 2, 1, 0, 0, 0])


**Training loop**

In [11]:
epoch = 120 # number of times to train on the entire dataset

# Loop through 120 training epochs
for e in range(epoch):
    optimizer.zero_grad() # clear gradients from the previous iteration
    outputs = func_model(X) # forward pass: predict outputs for X
    loss = criterion(outputs, Y) # calculate prediction error against Y

    loss.backward() # backpropagation: calculate gradients
    optimizer.step() # update model weights using the gradients

    # Print progress every 10 epochs
    if (e+1) % 10 == 0:
        print(f"Epoch [{e+1}/120], Loss: {loss.item():.3f}")

Epoch [10/120], Loss: 0.002
Epoch [20/120], Loss: 0.002
Epoch [30/120], Loss: 0.002
Epoch [40/120], Loss: 0.002
Epoch [50/120], Loss: 0.002
Epoch [60/120], Loss: 0.002
Epoch [70/120], Loss: 0.002
Epoch [80/120], Loss: 0.002
Epoch [90/120], Loss: 0.001
Epoch [100/120], Loss: 0.001
Epoch [110/120], Loss: 0.001
Epoch [120/120], Loss: 0.001
